# Step 2 — SQL Database Setup
**Project:** E-commerce Customer Segmentation  
**Input:** `online_retail_clean.csv` (from Step 1)  
**Goal:** Load clean data into a SQL database (star schema), then write RFM queries.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving online_retail_clean.csv to online_retail_clean.csv


In [2]:
import sqlite3
import pandas as pd

In [3]:
df = pd.read_csv('online_retail_clean.csv')

In [4]:
len(df)

805549

In [5]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,TotalRevenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


### 2. Why a star schema?
Instead of one giant flat table, we split data into:
- **fact_transactions** — every purchase (the big table, lots of rows)
- **dim_customers** — one row per customer (small lookup table)
- **dim_products** — one row per product (small lookup table)


It avoids repeating customer/product info on every single row and is a strong signal in interviews that you understand data modeling.

### 3. Build the dimension and fact tables in pandas

In [8]:
# dim_customers — one row per customer
dim_customers = df[['CustomerID', 'Country']].drop_duplicates(subset='CustomerID').reset_index(drop=True)


In [7]:
# dim_products — one row per product
dim_products = df[['StockCode', 'Description']].drop_duplicates(subset='StockCode').reset_index(drop=True)

In [9]:
# fact_transactions — every purchase, referencing the dimension tables by key
fact_transactions = df[['Invoice', 'CustomerID', 'StockCode', 'InvoiceDate', 'Quantity', 'Price', 'TotalRevenue']].copy()

In [10]:
len(dim_customers), len(dim_products), len(fact_transactions)

(5878, 4631, 805549)

### 4. Create the SQLite database and load the tables

In [12]:
# Creates a file called online_retail.db in the current folder
conn = sqlite3.connect('online_retail.db')


In [13]:
dim_customers.to_sql('dim_customers', conn, if_exists='replace', index=False)
dim_products.to_sql('dim_products', conn, if_exists='replace', index=False)
fact_transactions.to_sql('fact_transactions', conn, if_exists='replace', index=False)


805549

In [14]:
print('Database created: online_retail.db')
print()
print('Tables in database:')
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
for t in tables:
    count = conn.execute(f'SELECT COUNT(*) FROM {t[0]}').fetchone()[0]
    print(f'  {t[0]:<20} {count:>10,} rows')

Database created: online_retail.db

Tables in database:
  dim_customers             5,878 rows
  dim_products              4,631 rows
  fact_transactions       805,549 rows


### 5. Sanity check — run a simple query

top 5 countries by revenue.

In [15]:
query = """
SELECT
    c.Country,
    ROUND(SUM(f.TotalRevenue), 2) AS TotalRevenue,
    COUNT(DISTINCT f.Invoice) AS NumOrders
FROM fact_transactions f
JOIN dim_customers c ON f.CustomerID = c.CustomerID
GROUP BY c.Country
ORDER BY TotalRevenue DESC
LIMIT 5
"""

pd.read_sql(query, conn)

,Country,TotalRevenue,NumOrders
0,United Kingdom,14722423.67,33539
1,EIRE,622354.96,569
2,Netherlands,554232.34,228
3,Germany,432182.44,790
4,France,353579.39,610


### 6. The main query — RFM calculation
This is the most important SQL in the whole project. For every customer, it calculates:
- **Recency** — days since their last purchase
- **Frequency** — number of distinct orders
- **Monetary** — total amount spent

It uses a CTE (`WITH ... AS`) to first find the reference date (one day after the last transaction in the whole dataset), then computes the three metrics per customer.


In [16]:
rfm_query = """
WITH reference_date AS (
    SELECT DATE(MAX(InvoiceDate), '+1 day') AS ref_date
    FROM fact_transactions
)
SELECT
    CustomerID,
    CAST((JULIANDAY((SELECT ref_date FROM reference_date)) - JULIANDAY(MAX(InvoiceDate))) AS INTEGER) AS Recency,
    COUNT(DISTINCT Invoice) AS Frequency,
    ROUND(SUM(TotalRevenue), 2) AS Monetary
FROM fact_transactions
GROUP BY CustomerID
"""

rfm = pd.read_sql(rfm_query, conn)
print(f'RFM table built for {len(rfm):,} customers')
rfm.head(10)


RFM table built for 5,878 customers


,CustomerID,Recency,Frequency,Monetary
0,12346,325,12,77556.46
1,12347,2,8,5633.32
2,12348,75,5,2019.40
3,12349,18,4,4428.69
4,12350,310,1,334.40
5,12351,375,1,300.93
6,12352,36,10,2849.84
7,12353,204,2,406.76
8,12354,232,1,1079.40
9,12355,214,2,947.61


### 7. Check the RFM numbers make sense


In [17]:
rfm[['Recency', 'Frequency', 'Monetary']].describe()

,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000
mean,200.866791,6.289384,3018.616734
std,209.353961,13.009406,14737.731038
min,0.000000,1.000000,2.950000
25%,25.000000,1.000000,348.762500
50%,95.000000,3.000000,898.915000
75%,379.000000,7.000000,2307.090000
max,738.000000,398.000000,608821.650000


Quick read of these numbers:
- **Recency**: most customers bought somewhat recently (median ~95 days), but some haven't bought in 700+ days — these are likely "Lost" customers.
- **Frequency**: most customers ordered only a few times (median 3), while a few ordered 100+ times — these are your power users.
- **Monetary**: huge range — some customers spent under £10, one spent over £600,000. This is typical for retail data (a few big wholesale buyers).


### 8. Bonus query — monthly revenue trend (window function practice)


In [18]:
monthly_query = """
SELECT
    strftime('%Y-%m', InvoiceDate) AS Month,
    ROUND(SUM(TotalRevenue), 2) AS Revenue,
    LAG(ROUND(SUM(TotalRevenue), 2)) OVER (ORDER BY strftime('%Y-%m', InvoiceDate)) AS PrevMonthRevenue
FROM fact_transactions
GROUP BY Month
ORDER BY Month
"""

monthly = pd.read_sql(monthly_query, conn)
monthly['MoM_Growth_%'] = ((monthly['Revenue'] - monthly['PrevMonthRevenue']) / monthly['PrevMonthRevenue'] * 100).round(1)
monthly


,Month,Revenue,PrevMonthRevenue,MoM_Growth_%
0,2009-12,686654.16,NaN,NaN
1,2010-01,557319.06,686654.16,-18.8
2,2010-02,506371.07,557319.06,-9.1
3,2010-03,699608.99,506371.07,38.2
4,2010-04,594609.19,699608.99,-15.0
5,2010-05,599985.79,594609.19,0.9
6,2010-06,639066.58,599985.79,6.5
7,2010-07,591636.74,639066.58,-7.4
8,2010-08,604242.65,591636.74,2.1
9,2010-09,831615.00,604242.65,37.6


### 9. Bonus query — rank customers by revenue (window function)

In [19]:
rank_query = """
SELECT
    CustomerID,
    ROUND(SUM(TotalRevenue), 2) AS TotalSpend,
    RANK() OVER (ORDER BY SUM(TotalRevenue) DESC) AS SpendRank
FROM fact_transactions
GROUP BY CustomerID
ORDER BY SpendRank
LIMIT 10
"""

pd.read_sql(rank_query, conn)


,CustomerID,TotalSpend,SpendRank
0,18102,608821.65,1
1,14646,528602.52,2
2,14156,313946.37,3
3,14911,295972.63,4
4,17450,246973.09,5
5,13694,196482.81,6
6,17511,175603.55,7
7,16446,168472.50,8
8,16684,147142.77,9
9,12415,144458.37,10


### 10. Save the RFM table for Step 3 (clustering)

In [20]:
rfm.to_csv('rfm_table.csv', index=False)
print('Saved: rfm_table.csv')
print(f'{len(rfm):,} customers ready for clustering in Step 3')


Saved: rfm_table.csv
5,878 customers ready for clustering in Step 3


In [21]:
conn.close()
print('\nDatabase connection closed.')
print('Step 2 COMPLETE. Files ready: online_retail.db, rfm_table.csv')


Database connection closed.
Step 2 COMPLETE. Files ready: online_retail.db, rfm_table.csv


In [22]:
from google.colab import files
files.download('rfm_table.csv')
files.download('online_retail.db')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>